In [1]:
import pandas as pd
import numpy as np
import re, unicodedata, json
from itertools import combinations
from pathlib import Path
import plotly.graph_objects as go
import plotly.express as px
import plotly.io as pio
pio.templates.default = "seaborn" # or another preferred theme
pio.templates[pio.templates.default].layout.colorway = px.colors.qualitative.G10
px.defaults.color_continuous_scale = px.colors.sequential.Cividis_r
import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)
colorscale="haline"

# **Bloque A — Varios HTML (uno por plot) con dropdown por figura, ya “bonito”**

In [2]:
# =========================
# CONFIG
# =========================
SCOPUS_CSV_PATH = "scopus_export_Mar 1-2026_82a43998-7fb2-4ba9-b730-2716fe57e870.csv"
FACULTY_XLSX_PATH = "Base de Datos Scopus.xlsx"
FACULTY_SHEET = "Hoja1"

START_YEAR = 2022
OUT_ROOT = Path("utb_scopus_dashboard_single_pretty")
OUT_DIR = OUT_ROOT / "_ALL"

TOP_SCHOOLS = 18
TOP_AUTHORS = 20
TOP_PAIRS = 30
DOC_TYPES_ORDER = ["Article", "Conference", "Review", "Other"]

In [4]:
# =========================
# Helpers
# =========================
def _norm(s) -> str:
    if s is None or (isinstance(s, float) and np.isnan(s)):
        return ""
    s = str(s)
    s = unicodedata.normalize("NFKD", s)
    s = "".join(ch for ch in s if not unicodedata.combining(ch))
    s = s.lower()
    s = re.sub(r"\s+", " ", s).strip()
    return s

def _split_semicol(s):
    if s is None or (isinstance(s, float) and np.isnan(s)):
        return []
    xs = [x.strip() for x in str(s).split(";")]
    return [x for x in xs if x != ""]

def _parse_author_aff_entry(entry: str):
    if entry is None or (isinstance(entry, float) and np.isnan(entry)):
        return (None, None)
    entry = str(entry).strip()
    if entry == "" or _norm(entry) in ("nan", "null", "0"):
        return (None, None)
    parts = [p.strip() for p in entry.split(",", 2)]
    if len(parts) >= 3:
        last, first, aff = parts[0], parts[1], parts[2]
        name = f"{last}, {first}".strip(", ")
    elif len(parts) == 2:
        last, first = parts
        name = f"{last}, {first}".strip(", ")
        aff = None
    else:
        name = parts[0]
        aff = None
    if _norm(name) in ("", "nan", "null", "0"):
        name = None
    if isinstance(aff, str) and _norm(aff) in ("", "nan", "null", "0"):
        aff = None
    return name, aff

def _extract_authorid_from_url(x):
    if x is None or (isinstance(x, float) and np.isnan(x)):
        return None
    m = re.search(r"authorId=(\d+)", str(x))
    return m.group(1) if m else None

def _canon_author_id(v):
    if pd.isna(v):
        return None
    try:
        iv = int(v)
        return None if iv == 0 else str(iv)
    except Exception:
        s = str(v).strip()
        return None if s in ("", "0", "nan") else s

def _doc_type_bucket(dt) -> str:
    t = _norm(dt)
    if t == "":
        return "Other"
    if "review" in t:
        return "Review"
    if "conference" in t or "proceeding" in t:
        return "Conference"
    if "article" in t:
        return "Article"
    return "Other"

In [5]:
# =========================
# Load faculty (planta UTB)
# =========================
faculty_raw = pd.read_excel(FACULTY_XLSX_PATH, sheet_name=FACULTY_SHEET).copy()
faculty_raw["author_id_from_url"] = faculty_raw.get("SCOPUS", pd.Series([None]*len(faculty_raw))).apply(_extract_authorid_from_url)
faculty_raw["author_id"] = faculty_raw.get("ID SCOPUS", pd.Series([None]*len(faculty_raw))).apply(_canon_author_id)
faculty_raw.loc[faculty_raw["author_id"].isna(), "author_id"] = faculty_raw.loc[faculty_raw["author_id"].isna(), "author_id_from_url"]

faculty_valid = faculty_raw[faculty_raw["author_id"].notna()].copy()
faculty_valid["author_id"] = faculty_valid["author_id"].astype(str).str.strip()
faculty_valid = faculty_valid.drop_duplicates(subset=["author_id"]).reset_index(drop=True)

faculty_missing_id = faculty_raw[faculty_raw["author_id"].isna()].copy().reset_index(drop=True)

for col in ["DOCENTE", "ESCUELA"]:
    if col not in faculty_valid.columns:
        faculty_valid[col] = None

faculty_ids = set(faculty_valid["author_id"].tolist())
name_map = faculty_valid.set_index("author_id")["DOCENTE"].to_dict()
school_map = faculty_valid.set_index("author_id")["ESCUELA"].to_dict()

# =========================
# Load scopus + filter
# =========================
df = pd.read_csv(SCOPUS_CSV_PATH, encoding="utf-8-sig").copy()
df["Year"] = pd.to_numeric(df["Year"], errors="coerce")
df = df[df["Year"].notna()].copy()
df["Year"] = df["Year"].astype(int)
df = df[df["Year"] >= START_YEAR].copy()

if "Document Type" not in df.columns:
    df["Document Type"] = None
df["doc_type3"] = df["Document Type"].apply(_doc_type_bucket)

years = sorted(df["Year"].unique().tolist())
year_sels = ["ALL"] + [str(y) for y in years]

In [6]:
# =========================
# Build authors_long (from filtered papers)
# =========================
records = []
for _, r in df.iterrows():
    eid = r.get("EID")
    year = int(r.get("Year"))
    dt3 = r.get("doc_type3")

    ids = _split_semicol(r.get("Author(s) ID"))
    awas = _split_semicol(r.get("Authors with affiliations"))
    short = _split_semicol(r.get("Authors"))

    n = max(len(ids), len(awas), len(short))
    ids += [None] * (n - len(ids))
    awas += [None] * (n - len(awas))
    short += [None] * (n - len(short))

    for i in range(n):
        name, _aff = _parse_author_aff_entry(awas[i]) if awas[i] is not None else (None, None)
        if name is None and short[i] is not None and str(short[i]).strip() != "":
            name = str(short[i]).strip()

        author_id = str(ids[i]).strip() if ids[i] is not None else None
        if author_id in ("", "nan", "None"):
            author_id = None

        records.append(
            {
                "EID": eid,
                "Year": year,
                "doc_type3": dt3,
                "author_position": i + 1,
                "author_id": author_id,
                "author_name": name,
            }
        )

authors_long = pd.DataFrame.from_records(records)
authors_long = authors_long.dropna(subset=["author_id", "author_name"], how="all").reset_index(drop=True)
authors_long["author_id"] = authors_long["author_id"].astype(str).str.strip()

# Only faculty (planta)
utb_planta_long = authors_long[authors_long["author_id"].isin(faculty_ids)].copy()
utb_planta_long["DOCENTE"] = utb_planta_long["author_id"].map(name_map).fillna(utb_planta_long["author_id"])
utb_planta_long["ESCUELA"] = utb_planta_long["author_id"].map(school_map).fillna("ESCUELA (missing)")

# Unique-paper “credit” datasets
planta_school_papers = utb_planta_long[["EID", "Year", "ESCUELA", "doc_type3"]].drop_duplicates()
planta_author_papers = utb_planta_long[["EID", "Year", "author_id", "DOCENTE", "ESCUELA", "doc_type3"]].drop_duplicates()

# =========================
# Aggregators
# =========================
def _filter_year(df_in: pd.DataFrame, year_sel: str) -> pd.DataFrame:
    if year_sel == "ALL":
        return df_in
    return df_in[df_in["Year"] == int(year_sel)]

def school_doc_counts(year_sel: str):
    base = _filter_year(planta_school_papers, year_sel).copy()
    counts = base.groupby(["ESCUELA", "doc_type3"]).size().reset_index(name="n_docs")

    tot = counts.groupby("ESCUELA")["n_docs"].sum().reset_index(name="total")
    top = tot.sort_values("total", ascending=False).head(TOP_SCHOOLS)["ESCUELA"].tolist()

    counts["ESCUELA2"] = np.where(counts["ESCUELA"].isin(top), counts["ESCUELA"], "Other")
    counts2 = (
        counts.groupby(["ESCUELA2", "doc_type3"], as_index=False)["n_docs"]
        .sum()
        .rename(columns={"ESCUELA2": "ESCUELA"})
    )
    counts2["doc_type3"] = pd.Categorical(counts2["doc_type3"], categories=DOC_TYPES_ORDER, ordered=True)

    # orden por total DESC (para que el más grande quede abajo en barra horizontal)
    order = counts2.groupby("ESCUELA")["n_docs"].sum().sort_values(ascending=False).index.tolist()
    return counts2, order

def author_doc_counts(year_sel: str):
    base = _filter_year(planta_author_papers, year_sel).copy()

    # papers únicos por (autor, tipo)
    counts = (
        base.groupby(["author_id", "DOCENTE", "ESCUELA", "doc_type3"])
        .size()
        .reset_index(name="n_docs")
    )

    # total por autor (en el año seleccionado)
    tot = (
        counts.groupby(["author_id", "DOCENTE", "ESCUELA"], as_index=False)["n_docs"]
        .sum()
        .rename(columns={"n_docs": "total"})
    )

    # Top N autores (en el año seleccionado)
    top_ids = tot.sort_values("total", ascending=False).head(TOP_AUTHORS)["author_id"].tolist()
    counts = counts[counts["author_id"].isin(top_ids)].copy()

    counts["doc_type3"] = pd.Categorical(counts["doc_type3"], categories=DOC_TYPES_ORDER, ordered=True)

    # ORDEN (fijo y correcto): suma por DOCENTE y ordena por total del año
    tot2 = counts.groupby("DOCENTE", as_index=False)["n_docs"].sum()
    order = tot2.sort_values("n_docs", ascending=True)["DOCENTE"].tolist()  # ascending=True: mayor queda abajo

    return counts, order

def heatmap_school_doctype(year_sel: str):
    counts, school_order = school_doc_counts(year_sel)

    piv = counts.pivot_table(index="ESCUELA", columns="doc_type3", values="n_docs", fill_value=0)

    # Reindex de filas y columnas + fill_value (evita NaNs desde la fuente)
    piv = piv.reindex(index=school_order, columns=DOC_TYPES_ORDER, fill_value=0)

    # Forzar a numérico por si algo quedó como object
    piv = piv.apply(pd.to_numeric, errors="coerce").fillna(0).astype(int)

    # add totals row/col
    piv["TOTAL"] = piv.sum(axis=1)
    piv.loc["TOTAL"] = piv.sum(axis=0)

    # debug opcional
    # print(year_sel, "NaNs:", int(piv.isna().sum().sum()))

    return piv

# =========================
# Pairs faculty–faculty
# =========================
paper_fac = utb_planta_long[["EID", "Year", "author_id"]].drop_duplicates()

pairs_rows = []
for eid, g in paper_fac.groupby("EID"):
    y = int(g["Year"].iloc[0])
    ids = sorted(set(g["author_id"].astype(str).tolist()))
    if len(ids) < 2:
        continue
    for a, b in combinations(ids, 2):
        pairs_rows.append((y, a, b, eid))

pairs_df = pd.DataFrame(pairs_rows, columns=["Year", "author_id_a", "author_id_b", "EID"])
pairs_counts_year = (
    pairs_df.groupby(["Year", "author_id_a", "author_id_b"])
    .agg(n_shared=("EID", "nunique"))
    .reset_index()
)
pairs_counts_all = (
    pairs_df.groupby(["author_id_a", "author_id_b"])
    .agg(n_shared=("EID", "nunique"))
    .reset_index()
)

def top_pairs(year_sel: str) -> pd.DataFrame:
    if year_sel == "ALL":
        c = pairs_counts_all.copy()
    else:
        y = int(year_sel)
        c = pairs_counts_year[pairs_counts_year["Year"] == y].drop(columns=["Year"]).copy()

    if c.empty:
        return c

    c["DOCENTE_A"] = c["author_id_a"].map(name_map).fillna(c["author_id_a"])
    c["DOCENTE_B"] = c["author_id_b"].map(name_map).fillna(c["author_id_b"])
    c["ESCUELA_A"] = c["author_id_a"].map(school_map)
    c["ESCUELA_B"] = c["author_id_b"].map(school_map)
    c["pair"] = c["DOCENTE_A"] + " — " + c["DOCENTE_B"]

    return c.sort_values("n_shared", ascending=False).head(TOP_PAIRS).copy()

In [7]:
# =========================
# Plotly
# =========================
import plotly.graph_objects as go
import plotly.express as px
import plotly.io as pio

OUT_DIR.mkdir(parents=True, exist_ok=True)

def _base_layout(title: str, width: int = 1200, height: int = 700):
    return dict(
        title={"text": title},
        width=width,
        height=height,
        font=dict(size=16),
        margin=dict(l=110, r=40, t=110, b=80),
        legend=dict(
            title="Tipo de documento",
            orientation="h",
            x=0.0, xanchor="left",
            y=-0.18, yanchor="top",
        ),
    )

def _base_layout_enhanced(title: str, width: int = 1200, height: int = 700):
    return dict(
        title={"text": title},
        width=width,
        height=height,
        font=dict(size=16),
        margin=dict(l=110, r=70, t=110, b=90),
        xaxis=dict(title_font=dict(size=18), tickfont=dict(size=12)),
        yaxis=dict(title_font=dict(size=18), tickfont=dict(size=12)),
    )

def _dropdown(fig, buttons, x=0.99, y=1.08, label_x=0.85, label_y=1.08):
    fig.update_layout(
        updatemenus=[dict(
            type="dropdown",
            x=x, y=y,
            xanchor="right", yanchor="top",
            direction="down",
            buttons=buttons,
            bgcolor="white",
            bordercolor="rgba(0,0,0,0.2)",
            borderwidth=1,
        )]
    )
    fig.add_annotation(
        text="Selecciona el año de análisis:",
        x=label_x, y=label_y, xref="paper", yref="paper",
        xanchor="right", yanchor="top",
        showarrow=False,
        font=dict(size=14),
    )

def write_fig(fig, path: Path):
    pio.write_html(fig, file=str(path), include_plotlyjs="cdn", full_html=True)

# ---- 1) Overall papers per year (STACKED by type), HORIZONTAL + labels ES ----
overall_year_doc = (
    utb_planta_long[["EID", "Year", "doc_type3"]].drop_duplicates()
    .groupby(["Year", "doc_type3"]).size().reset_index(name="n_docs")
)
overall_year_doc["doc_type3"] = pd.Categorical(
    overall_year_doc["doc_type3"], categories=DOC_TYPES_ORDER, ordered=True
)
overall_year_doc = overall_year_doc.sort_values(["Year", "doc_type3"])

fig_overall = px.bar(
    overall_year_doc,
    y="Year",
    x="n_docs",
    color="doc_type3",
    orientation="h",
    barmode="stack",
    hover_data={"n_docs": True, "Year": True, "doc_type3": True},
    labels={"n_docs": "Número de documentos", "Year": "Años", "doc_type3": "Tipo de documento"},
    title=f"Documentos por año (>= {START_YEAR}) – apilado por tipo",
)
fig_overall.update_layout(**_base_layout(fig_overall.layout.title.text))
fig_overall.update_xaxes(title_text="Número de documentos", title_font=dict(size=18), tickfont=dict(size=14))
fig_overall.update_yaxes(title_text="Años", title_font=dict(size=18), tickfont=dict(size=14))

# Totales al final (por año)
tot_year = overall_year_doc.groupby("Year")["n_docs"].sum().reset_index()
pad = max(tot_year["n_docs"].max() * 0.01, 1)
fig_overall.add_trace(go.Scatter(
    x=(tot_year["n_docs"] + pad).tolist(),
    y=tot_year["Year"].tolist(),
    mode="text",
    text=tot_year["n_docs"].astype(int).astype(str).tolist(),
    textfont=dict(size=13),
    showlegend=False,
    hoverinfo="skip",
    cliponaxis=False,
))
write_fig(fig_overall, OUT_DIR / "overall_papers_per_year_STACKED_by_type_horizontal.html")

# ---- 2) Papers by Escuela (STACKED) with dropdown + totals ----
fig_school = go.Figure()
trace_meta = []  # (year_sel, tag, trace_idx)
school_orders = {}

for ys in year_sels:
    counts, order = school_doc_counts(ys)
    school_orders[ys] = order

    for dt in DOC_TYPES_ORDER:
        sub = counts[counts["doc_type3"].astype(str) == dt].copy()
        base = pd.DataFrame({"ESCUELA": order})
        sub = base.merge(sub[["ESCUELA", "n_docs"]], on="ESCUELA", how="left").fillna({"n_docs": 0})

        fig_school.add_trace(go.Bar(
            x=sub["n_docs"].tolist(),
            y=sub["ESCUELA"].tolist(),
            name=dt,
            orientation="h",
            visible=(ys == "ALL"),
            hovertemplate="Escuela=%{y}<br>Tipo=" + dt + "<br>Documentos=%{x}<extra></extra>",
        ))
        trace_meta.append((ys, dt, len(fig_school.data) - 1))

    tot = counts.groupby("ESCUELA")["n_docs"].sum().reindex(order, fill_value=0)
    pad = max(tot.max() * 0.01, 1)
    fig_school.add_trace(go.Scatter(
        x=(tot.values + pad).tolist(),
        y=order,
        mode="text",
        text=[str(int(v)) for v in tot.values],
        textfont=dict(size=12),
        showlegend=False,
        hoverinfo="skip",
        visible=(ys == "ALL"),
        cliponaxis=False,
    ))
    trace_meta.append((ys, "__TOTAL__", len(fig_school.data) - 1))

buttons = []
for ys in year_sels:
    vis = [False] * len(fig_school.data)
    for (y2, tag, idx) in trace_meta:
        if y2 == ys:
            vis[idx] = True

    buttons.append(dict(
        label=str(ys),
        method="update",
        args=[
            {"visible": vis},
            {
                "title": {"text": f"Documentos por Escuela – apilado por tipo ({ys})"},
                "yaxis": {"categoryorder": "array", "categoryarray": school_orders[ys]},
                "xaxis": {"title": "Número de documentos"},
            },
        ],
    ))

fig_school.update_layout(**_base_layout("Documentos por Escuela – apilado por tipo"), barmode="stack")
fig_school.update_xaxes(title_text="Número de documentos", title_font=dict(size=18), tickfont=dict(size=14))
fig_school.update_yaxes(
    title_text="Escuela",
    title_font=dict(size=18),
    tickfont=dict(size=12),
    categoryorder="array",
    categoryarray=school_orders["ALL"],
)
_dropdown(fig_school, buttons)
write_fig(fig_school, OUT_DIR / "papers_by_escuela_STACKED_by_type_YEAR_dropdown.html")

# ---- 3) Top authors STACKED with dropdown + totals ----
fig_auth = go.Figure()
trace_meta_a = []  # (year_sel, tag, trace_idx)
author_orders = {}

for ys in year_sels:
    counts, order = author_doc_counts(ys)
    author_orders[ys] = order

    for dt in DOC_TYPES_ORDER:
        sub = counts[counts["doc_type3"].astype(str) == dt].copy()
        base = pd.DataFrame({"DOCENTE": order})
        sub = base.merge(sub[["DOCENTE", "n_docs"]], on="DOCENTE", how="left").fillna({"n_docs": 0})

        fig_auth.add_trace(go.Bar(
            x=sub["n_docs"].tolist(),
            y=sub["DOCENTE"].tolist(),
            name=dt,
            orientation="h",
            visible=(ys == "ALL"),
            hovertemplate="Autor=%{y}<br>Tipo=" + dt + "<br>Documentos=%{x}<extra></extra>",
        ))
        trace_meta_a.append((ys, dt, len(fig_auth.data) - 1))

    tot = counts.groupby("DOCENTE")["n_docs"].sum().reindex(order, fill_value=0)
    pad = max(tot.max() * 0.01, 1)
    fig_auth.add_trace(go.Scatter(
        x=(tot.values + pad).tolist(),
        y=order,
        mode="text",
        text=[str(int(v)) for v in tot.values],
        textfont=dict(size=12),
        showlegend=False,
        hoverinfo="skip",
        visible=(ys == "ALL"),
        cliponaxis=False,
    ))
    trace_meta_a.append((ys, "__TOTAL__", len(fig_auth.data) - 1))

buttons_a = []
for ys in year_sels:
    vis = [False] * len(fig_auth.data)
    for (y2, tag, idx) in trace_meta_a:
        if y2 == ys:
            vis[idx] = True

    buttons_a.append(dict(
        label=str(ys),
        method="update",
        args=[
            {"visible": vis},
            {
                "title": {"text": f"Top {TOP_AUTHORS} autores – apilado por tipo ({ys})"},
                "yaxis": {"categoryorder": "array", "categoryarray": author_orders[ys]},
                "xaxis": {"title": "Número de documentos"},
            },
        ],
    ))

fig_auth.update_layout(**_base_layout(f"Top {TOP_AUTHORS} autores – apilado por tipo"), barmode="stack")
fig_auth.update_xaxes(title_text="Número de documentos", title_font=dict(size=18), tickfont=dict(size=14))
fig_auth.update_yaxes(
    title_text="Autor",
    title_font=dict(size=18),
    tickfont=dict(size=12),
    categoryorder="array",
    categoryarray=author_orders["ALL"],
)
_dropdown(fig_auth, buttons_a)
write_fig(fig_auth, OUT_DIR / "top_authors_STACKED_by_type_YEAR_dropdown.html")

# ---- 4) Heatmap Escuela × Tipo with dropdown + TOTAL row/col ----
fig_heat = go.Figure()
heat_orders = {}

for ys in year_sels:
    piv = heatmap_school_doctype(ys)
    heat_orders[ys] = list(piv.index)

    fig_heat.add_trace(go.Heatmap(
        z=piv.values.tolist(),
        x=list(piv.columns),
        y=list(piv.index),
        visible=(ys == "ALL"),
        colorbar=dict(
            title="Número de<br>documentos", # <br> ayuda si el título es largo
            orientation="v",               # "v" para vertical (por defecto)
            thickness=20,                  # Ancho de la barra en píxeles
            len=0.75,                      # Altura de la barra (0 a 1) respecto al plot
            yanchor="middle",              # Centra la barra verticalmente
            y=0.5
        ),
        hovertemplate="Escuela=%{y}<br>Tipo=%{x}<br>Documentos=%{z}<extra></extra>",
    ))

buttons_h = []
for i, ys in enumerate(year_sels):
    vis = [False] * len(fig_heat.data)
    vis[i] = True
    buttons_h.append(dict(
        label=str(ys),
        method="update",
        args=[
            {"visible": vis},
            {
                "title": {"text": f"Heatmap Escuela × Tipo (+ totales) – {ys}"},
                "yaxis": {"categoryorder": "array", "categoryarray": heat_orders[ys]},
            },
        ],
    ))

fig_heat.update_layout(**_base_layout_enhanced("Heatmap Escuela × Tipo (+ totales)"))
fig_heat.update_xaxes(title_text="Tipo de documento")
fig_heat.update_yaxes(
    title_text="Escuela",
    categoryorder="array",
    categoryarray=heat_orders["ALL"],
)
_dropdown(fig_heat, buttons_h, y=1.08, label_y=1.08)
write_fig(fig_heat, OUT_DIR / "heatmap_escuela_doctype_YEAR_dropdown.html")

# ---- 5) Top pairs faculty–faculty with dropdown (no totals needed) ----
fig_pairs = go.Figure()
pair_orders = {}

for ys in year_sels:
    tp = top_pairs(ys)
    if tp.empty:
        x, y = [], []
    else:
        y = tp["pair"].iloc[::-1].tolist()
        x = tp["n_shared"].iloc[::-1].tolist()

    pair_orders[ys] = y

    fig_pairs.add_trace(go.Bar(
        x=x,
        y=y,
        orientation="h",
        visible=(ys == "ALL"),
        hovertemplate="Par=%{y}<br>Documentos compartidos=%{x}<extra></extra>",
        showlegend=False,
    ))

buttons_p = []
for i, ys in enumerate(year_sels):
    vis = [False] * len(fig_pairs.data)
    vis[i] = True
    buttons_p.append(dict(
        label=str(ys),
        method="update",
        args=[
            {"visible": vis},
            {
                "title": {"text": f"Top {TOP_PAIRS} pares (planta–planta) – {ys}"},
                "yaxis": {"categoryorder": "array", "categoryarray": pair_orders[ys]},
            },
        ],
    ))

fig_pairs.update_layout(title={"text": f"Top {TOP_PAIRS} pares (planta–planta)"}, font=dict(size=16),
                        margin=dict(l=140, r=40, t=110, b=80))
fig_pairs.update_xaxes(title_text="Documentos compartidos", title_font=dict(size=18), tickfont=dict(size=14))
fig_pairs.update_yaxes(
    title_text="Par",
    title_font=dict(size=18),
    tickfont=dict(size=12),
    categoryorder="array",
    categoryarray=pair_orders["ALL"],
)
_dropdown(fig_pairs, buttons_p)
write_fig(fig_pairs, OUT_DIR / "top_pairs_faculty_faculty_YEAR_dropdown.html")

In [8]:
# ---- tables + index ----
tables_path = OUT_DIR / "tables.xlsx"
with pd.ExcelWriter(tables_path, engine="openpyxl") as w:
    overall_year_doc.rename(columns={"n_docs":"n_documentos"}).to_excel(w, sheet_name="overall_year_docType", index=False)
    planta_school_papers.to_excel(w, sheet_name="planta_school_papers", index=False)
    planta_author_papers.to_excel(w, sheet_name="planta_author_papers", index=False)
    pairs_counts_year.to_excel(w, sheet_name="pairs_counts_year", index=False)
    pairs_counts_all.to_excel(w, sheet_name="pairs_counts_all", index=False)
    faculty_missing_id.to_excel(w, sheet_name="faculty_missing_id", index=False)

In [12]:
index_html = """<!doctype html>
<html>
<head>
  <meta charset="utf-8">
  <title>UTB Scopus Dashboard (>= __START_YEAR__)</title>
  <style>
    body { font-family: Arial, sans-serif; margin: 68px; line-height: 1.35; background-color: #f0f7ff;}
    .meta { max-width: 1100px; }
    .note { background: #FEFCE8; padding: 12px 14px; border-radius: 10px; border: 1px solid rgba(0,0,0,0.08); }
    ul { margin-top: 10px; }
    li { margin: 8px 0; }
    h2 { margin-bottom: 6px; }
    h3 { margin-top: 18px; }
    code { background: #f1f1f1; padding: 1px 4px; border-radius: 6px; font-family: Arial, sans-serif; }

    /* Cards grid */
    .cards-grid{
      display:grid;
      grid-template-columns: repeat(3, minmax(240px, 1fr));
      gap: 14px;
      margin-top: 10px;
      max-width: 1100px;
    }
    @media (max-width: 980px){
      .cards-grid{ grid-template-columns: repeat(2, minmax(240px, 1fr)); }
    }
    @media (max-width: 640px){
      .cards-grid{ grid-template-columns: 1fr; }
    }
    
    .card-link{
      display: block;        /* <-- hace que el <a> sea un bloque en la grid */
      height: 100%;          /* <-- evita “colapsos” raros */
      text-decoration:none;
      color: inherit;
    }


    
    .card{
      background: #E0E7FF;
      box-sizing: border-box; /* <-- evita que padding/border rompan el layout */
      border: 1px solid rgba(0,0,0,0.10);
      border-radius: 12px;
      padding: 14px 14px 12px 14px;
      box-shadow: 0 1px 2px rgba(0,0,0,0.04);
      transition: transform 0.08s ease, box-shadow 0.08s ease, border-color 0.08s ease;
      height: 100%;
    }
    .card:hover{
      transform: translateY(-1px);
      box-shadow: 0 6px 16px rgba(0,0,0,0.08);
      border-color: rgba(0,0,0,0.18);
    }
    .card-title{
      font-size: 16px;
      font-weight: 700;
      margin: 0 0 6px 0;
    }
    .card-desc{
      font-size: 13px;
      margin: 0;
      color: rgba(0,0,0,0.72);
    }
    .card-meta{
      margin-top: 10px;
      font-size: 12px;
      color: rgba(0,0,0,0.55);
    }
    .tag{
      display:inline-block;
      padding: 2px 8px;
      border-radius: 999px;
      border: 1px solid rgba(0,0,0,0.10);
      background: rgba(0,0,0,0.02);
      margin-right: 6px;
      margin-top: 6px;
    }
  </style>
</head>

<body>
<div class="meta">
  <h1>UTB Scopus Dashboard (&gt;= __START_YEAR__)</h2>
  <p>
    Este panel presenta una caracterización bibliométrica de la producción científica asociada a docentes de planta
    de la Universidad Tecnológica de Bolívar (UTB) a partir de un export de Scopus. La metodología integra:
    (i) una base maestra de docentes de planta con su <em>Scopus Author ID</em> (cuando está disponible),
    (ii) un archivo CSV exportado desde Scopus con metadatos por documento (p. ej., EID, año, tipo de documento y autores),
    y (iii) un procedimiento de normalización y cruce para identificar, dentro de cada documento, qué autores pertenecen a la planta UTB.
    Para cada documento se reconstruye la lista de autores y su posición de autoría; luego se filtran únicamente aquellos autores
    cuyo <em>Author ID</em> coincide con la lista de planta. Los conteos se realizan sobre <strong>documentos únicos</strong>
    (identificados por EID), evitando dobles conteos por múltiples apariciones del mismo autor. La desagregación por
    <strong>Escuela</strong> asigna crédito a una Escuela cuando al menos un docente de esa Escuela participa como autor en el documento
    (por lo que un mismo documento puede contribuir a más de una Escuela si existe coautoría inter-escuelas). Adicionalmente, el
    “Document Type” de Scopus se agrupa en categorías operativas (Article, Conference, Review, Other) para analizar la composición
    de la producción. Finalmente, los patrones de colaboración se resumen mediante pares de coautoría entre docentes de planta,
    contabilizando el número de documentos compartidos por par. En conjunto, los gráficos permiten explorar tendencias anuales,
    distribución por Escuela, perfiles de autoría y estructura de colaboración con criterios reproducibles y auditables.
  </p>

  <div class="note">
    <strong>Notas metodológicas</strong>
    <ul>
      <li><strong>Unidad de conteo:</strong> “Número de documentos” corresponde a <strong>papers únicos</strong> (conteo por <code>EID</code>), no a apariciones de autor.</li>
      <li><strong>Crédito por Escuela:</strong> una Escuela recibe crédito si al menos un docente de esa Escuela aparece como autor en el paper (por eso un paper puede contar en más de una Escuela si hay colaboración).</li>
      <li><strong>Tipos de documento:</strong> se agrupan en <em>Article</em>, <em>Conference</em>, <em>Review</em> y <em>Other</em> según “Document Type” del export.</li>
      <li><strong>Filtro temporal:</strong> se consideran años desde <strong>__START_YEAR__</strong> en adelante.</li>
    </ul>
    <p><strong>Cómo usar:</strong> en cada gráfico con dropdown, selecciona el año (o <em>ALL</em>) para actualizar la vista.</p>
  </div>

  <h2>Gráficos y Estadísticas</h2>
<strong>Nota Importante:</strong>
<p>Para la construcción de estas estadísticas y cuadros se hace la aclaratoria que todo esto está actualizado a fecha de <strong>__ACTU__</strong>.</p>

    
  <div class="cards-grid">
    <a class="card-link" href="overall_papers_per_year_STACKED_by_type_horizontal.html">
      <div class="card">
        <div class="card-title">Documentos por año</div>
        <p class="card-desc">Serie temporal global (documentos únicos), apilada por tipo de documento.</p>
        <div class="card-meta">
          <span class="tag">Global</span>
          <span class="tag">Tipo</span>
        </div>
      </div>
    </a>

    <a class="card-link" href="papers_by_escuela_STACKED_by_type_YEAR_dropdown.html">
      <div class="card">
        <div class="card-title">Documentos por Escuela</div>
        <p class="card-desc">Barras apiladas por tipo, con totales al final de cada barra. Incluye dropdown por año.</p>
        <div class="card-meta">
          <span class="tag">Escuela</span>
          <span class="tag">Tipo</span>
          <span class="tag">Dropdown</span>
        </div>
      </div>
    </a>

    <a class="card-link" href="top_authors_STACKED_by_type_YEAR_dropdown.html">
      <div class="card">
        <div class="card-title">Top autores</div>
        <p class="card-desc">Ranking de docentes por número de documentos únicos. Apilado por tipo, con dropdown por año.</p>
        <div class="card-meta">
          <span class="tag">Autoría</span>
          <span class="tag">Tipo</span>
          <span class="tag">Dropdown</span>
        </div>
      </div>
    </a>

    <a class="card-link" href="heatmap_escuela_doctype_YEAR_dropdown.html">
      <div class="card">
        <div class="card-title">Heatmap Escuela × Tipo</div>
        <p class="card-desc">Matriz Escuela × tipo, con totales por fila y columna. Incluye dropdown por año.</p>
        <div class="card-meta">
          <span class="tag">Heatmap</span>
          <span class="tag">Totales</span>
          <span class="tag">Dropdown</span>
        </div>
      </div>
    </a>

    <a class="card-link" href="top_pairs_faculty_faculty_YEAR_dropdown.html">
      <div class="card">
        <div class="card-title">Top pares planta–planta</div>
        <p class="card-desc">Pares de coautoría con más documentos compartidos. Incluye dropdown por año.</p>
        <div class="card-meta">
          <span class="tag">Colaboración</span>
          <span class="tag">Pares</span>
          <span class="tag">Dropdown</span>
        </div>
      </div>
    </a>

    <a class="card-link" href="tables.xlsx">
      <div class="card">
        <div class="card-title">Tablas (Excel)</div>
        <p class="card-desc">Archivo <code>tables.xlsx</code> con tablas intermedias para auditoría, pivots y reportes.</p>
        <div class="card-meta">
          <span class="tag">Export</span>
          <span class="tag">Auditoría</span>
        </div>
      </div>
    </a>
  </div>

<h3>Acaratoria:</h3>
<p class="note">
  <strong>Alcance e interpretación:</strong>
  Este tablero es un ejercicio técnico y personal de análisis bibliométrico basado en un export puntual de Scopus y una lista interna de docentes de planta.
  Por lo tanto, los resultados son <em>referenciales</em> y pueden diferir de cifras institucionales oficiales (por cobertura del export, actualización de perfiles en Scopus,
  homónimos/duplicados de Author ID, y reglas de conteo adoptadas). Este material no constituye un reporte oficial ni representa una posición institucional de la UTB;
  su propósito es exploratorio y de apoyo para ver tendencias y posibles inconsistencias de registro.
</p>
  
<div class="note">
    <strong>Créditos</strong>
    <ul>
      <li>Desarrollado por <strong>D. Sierra-Porta</strong> &copy; 2026 - Universidad Tecnológica de Bolívar</li>
    </ul>
  </div>

  
  <div class="spacer"></div> <!-- Espacio extra al final -->
  
</div>
</body>
</html>
"""

index_html = index_html.replace("__START_YEAR__", str(START_YEAR))
index_html = index_html.replace("__ACTU__", str(Actu))

(OUT_DIR / "index.html").write_text(index_html, encoding="utf-8")
print(f"Wrote: {OUT_DIR / 'index.html'}")

Wrote: utb_scopus_dashboard_single_pretty/_ALL/index.html


In [10]:
print("DONE ✅")
print(f"Open: {OUT_DIR / 'index.html'}")

DONE ✅
Open: utb_scopus_dashboard_single_pretty/_ALL/index.html


In [15]:
from pathlib import Path

FOLDER = "utb_scopus_dashboard_single_pretty"

readme_md = f"""# UTB Scopus Dashboard (>= __START_YEAR__)

Este repositorio contiene un tablero interactivo (HTML) y tablas (Excel) para explorar la producción científica asociada a docentes de planta de la
Universidad Tecnológica de Bolívar (UTB) a partir de un export de Scopus.

> **Actualización:** estas estadísticas y gráficos están actualizados a fecha de **__ACTU__**.

## Metodología (resumen)

- **Fuente de datos:** export CSV desde Scopus (EID, año, tipo de documento, autores).
- **Vinculación a planta UTB:** cruce por **Scopus Author ID** contra una base maestra interna de docentes de planta.
- **Unidad de conteo:** documentos únicos por **EID** (evita dobles conteos por múltiples apariciones del mismo autor).
- **Crédito por Escuela:** una Escuela recibe crédito si al menos un docente de esa Escuela aparece como autor en el documento (un documento puede contar en más de una Escuela si hay coautoría inter-escuelas).
- **Tipos de documento:** agrupación operativa en *Article*, *Conference*, *Review* y *Other* según “Document Type”.

## Abrir el tablero

Abre el archivo principal:

- **Página principal (índice):** `{FOLDER}/_ALL/index.html`

## Gráficos y estadísticas

- **Documentos por año (apilado por tipo):** `{FOLDER}/_ALL/overall_papers_per_year_STACKED_by_type_horizontal.html`
- **Documentos por Escuela (apilado + dropdown por año):** `{FOLDER}/_ALL/papers_by_escuela_STACKED_by_type_YEAR_dropdown.html`
- **Top autores (apilado + dropdown por año):** `{FOLDER}/_ALL/top_authors_STACKED_by_type_YEAR_dropdown.html`
- **Heatmap Escuela × Tipo (con totales + dropdown por año):** `{FOLDER}/_ALL/heatmap_escuela_doctype_YEAR_dropdown.html`
- **Top pares planta–planta (dropdown por año):** `{FOLDER}/_ALL/top_pairs_faculty_faculty_YEAR_dropdown.html`

## Tablas (Excel)

- **Descargar tablas:** `{FOLDER}/_ALL/tables.xlsx`

## Aclaratoria

Este tablero es un ejercicio técnico y personal de análisis bibliométrico basado en un export puntual de Scopus y una lista interna de docentes de planta.
Los resultados son **referenciales** y pueden diferir de cifras institucionales oficiales (cobertura del export, actualización de perfiles, homónimos/duplicados de Author ID, reglas de conteo).
Este material no constituye un reporte oficial ni representa una posición institucional de la UTB.

## Créditos
Desarrollado por **D. Sierra-Porta** © 2026 — Universidad Tecnológica de Bolívar
"""

# Reemplazos
readme_md = readme_md.replace("__START_YEAR__", str(START_YEAR)).replace("__ACTU__", str(Actu))

# Guardar en la raíz del repo
Path("README.md").write_text(readme_md, encoding="utf-8")
print("Wrote README.md")

Wrote README.md


In [16]:
readme_md = readme_md.replace("__START_YEAR__", str(START_YEAR)).replace("__ACTU__", str(Actu))
Path("README.md").write_text(readme_md, encoding="utf-8")

2574